### Introduction to Evolutionary Algorithm Maze Solver
This notebook demonstrates a search heuristic inspired by Charles Darwin's theory of natural evolution. It uses a Genetic Algorithm (GA) to find a path through a grid-based maze, navigating walls to find a goal state.

---

#### ✍️ My Explanation
### Why this representation of a maze was chosen

A 2D grid (numpy array) where `0` = open path and `1` = wall is a natural representation for a maze:

- Mazes are inherently grid-like spaces with individual cells and obstacles that the navigator cannot go through.
- Positions as `(row, col)` tuples match the very numpy array we're using which makes indexing and movement (up/down/left/right) straightforward.
- Manhattan distance (`|x1−x2| + |y1−y2|`) is a fast for grid pathfinding and works well as a fitness signal or guidance term inside a genetic algorithm. This is something we learned in DCS229 when learning about Djikstra's https://www.geeksforgeeks.org/data-science/manhattan-distance/

This keeps the environment simple and focused so the evolutionary algorithm can concentrate on evolving path-finding behavior rather than spending resources on a more complex continuous or graph-based representation. Evolutionary algorithms already use lots of resources, dependent on the scale of the operation, and we'd prefer to use them wisely.

### How encapsulating the maze in a class benefits the architecture

Putting the grid, start, goal, and spatial rules inside the `Maze` class is a the application of **encapsulation** and **abstraction** of Object-Oriented Programming:

- **Bundles state + behavior**: The data (grid, dimensions, start/goal) and the rules that operate on it (`is_valid_position`, `get_distance`) live together. Users never touch the raw grid directly, only looking at the data, never edit.
- **Hides implementation details**: The rest of the system (Individual, Population, Selection, Crossover, EvolutionEngine, etc.) only needs to “ask the Maze if this position is valid” or “what is the distance to the goal.” It does not care whether the grid is, that would be unnecessary implementation, and again wasted resources.
- **Clean separation of concerns**: The evolutionary algorithm owns the search mechanisms (genotypes, variation, selection). The Maze owns the problem  (what a valid path looks like, how fitness relates to the environment). This mirrors the genotype/phenotype split in DeJong’s framework.
- **Easier testing and reuse**: You can unit-test the Maze independently, swap in different mazes (or even different environment classes) without changing the GA code, and keep the evolutionary operators polymorphic and reusable.
- **Supports modularity and future extension**: If you later want a different heuristic, a continuous maze, or a multi-agent environment, you can change or subclass `Maze` while the core EvolutionEngine stays the same.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

class Maze:
    def __init__(self, grid, start, goal):
        """
        grid: 2D list/array where 0 is open path and 1 is a wall.
        start: tuple (row, col) representing start position
        goal: tuple (row, col) representing goal position
        """
        self.grid = np.array(grid)
        self.start = start
        self.goal = goal
        self.rows = len(grid)
        self.cols = len(grid[0])

    def is_valid_position(self, row, col):
        if 0 <= row < self.rows and 0 <= col < self.cols:
            return self.grid[row][col] == 0
        return False

    def get_distance(self, pos1, pos2):
        # Manhattan distance
        return abs(pos1[0] - pos2[0]) + abs(pos1[1] - pos2[1])

### 1. The Maze Environment
#### How it works:
* **`grid`**: A 2D list where `0` represents a traversable path cell and `1` represents a wall.
* **`is_valid_position(row, col)`**: Checks boundaries and ensures the coordinates do not overlap with a wall.
* **`get_distance(pos1, pos2)`**: Calculates Manhattan distance ($|x_1 - x_2| + |y_1 - y_2|$) from the candidate position to the goal. This calculation serves as a heuristic.


In [ ]:
class Individual:
    # Map directions to grid coordinates changes
    DIRECTIONS = {
        'U': (-1, 0),
        'D': (1, 0),
        'L': (0, -1),
        'R': (0, 1)
    }
    DIR_LIST = ['U', 'D', 'L', 'R']

    def __init__(self, maze, path_length, genes=None):
        self.maze = maze
        self.path_length = path_length
        # Genes are a list of directions (e.g., ['U', 'R', 'R', 'D'])
        if genes is None:
            self.genes = [random.choice(self.DIR_LIST) for _ in range(path_length)]
        else:
            self.genes = genes
        self.fitness = 0.0
        self.final_position = maze.start
        self.steps_taken = 0
        self.evaluate()

    def evaluate(self):
        """
        Simulates the individual's path through the maze and calculates its fitness.
        """
        row, col = self.maze.start
        visited_positions = {(row, col)}  # Set of unique visited positions
        self.steps_taken = 0

        for direction in self.genes:
            dr, dc = self.DIRECTIONS[direction]
            next_row, next_col = row + dr, col + dc

            if self.maze.is_valid_position(next_row, next_col):
                row, col = next_row, next_col
                visited_positions.add((row, col))
                self.steps_taken += 1
                if (row, col) == self.maze.goal:
                    break # Reached the goal early
            else:
                # Path hit a wall or edge, the step is ignored but costs an action
                self.steps_taken += 1

        self.final_position = (row, col)

        # Calculate fitness
        dist_to_goal = self.maze.get_distance(self.final_position, self.maze.goal)

        # High fitness for reaching the goal, penalty for distance and steps taken
        if self.final_position == self.maze.goal:
            self.fitness = 2000.0 + (self.path_length - self.steps_taken)
        else:
            # Reward exploration (unique cells visited) + proximity to goal
            # This helps the algorithm find paths around long walls instead of getting trapped
            exploration_bonus = len(visited_positions) * 1.5
            self.fitness = exploration_bonus + (100.0 / (dist_to_goal + 1.0))

    def mutate(self, mutation_rate):
        for i in range(len(self.genes)):
            if random.random() < mutation_rate:
                self.genes[i] = random.choice(self.DIR_LIST)
        self.evaluate()

    @classmethod
    def crossover(cls, parent1, parent2):
        """Single-point crossover"""
        crossover_point = random.randint(1, len(parent1.genes) - 1)
        child1_genes = parent1.genes[:crossover_point] + parent2.genes[crossover_point:]
        child2_genes = parent2.genes[:crossover_point] + parent1.genes[crossover_point:]

        child1 = cls(parent1.maze, parent1.path_length, child1_genes)
        child2 = cls(parent2.maze, parent2.path_length, child2_genes)
        return child1, child2

### 2. The Individual Representation (Genes and Evaluation)
#### How it works:
* **`genes`**: A fixed-length string of directions ('U', 'D', 'L', 'R') representing candidate movement instructions.
* **`evaluate()`**: Iterates through the genes step-by-step. If a step is invalid (hits a wall or edge), it is ignored but still counts as a step taken. If the goal is reached, simulation ends early.
* **`fitness`**: Computes the score of an individual. Reaching the goal yields a massive base reward (2000), penalized by the steps taken. If the goal is missed, it receives an exploration bonus based on unique visited cells plus a proximity reward based on the inverse Manhattan distance to the goal.
* **`mutate(mutation_rate)`**: Traverses genes and randomly swaps elements based on a small probability.
* **`crossover(parent1, parent2)`**: Combines genetic information of two parents around a random split index (single-point crossover) to produce two child individuals.

In [ ]:
class Population:
    def __init__(self, maze, pop_size, path_length, mutation_rate=0.05, elitism_size=2):
        self.maze = maze
        self.pop_size = pop_size
        self.path_length = path_length
        self.mutation_rate = mutation_rate
        self.elitism_size = elitism_size
        self.individuals = [Individual(maze, path_length) for _ in range(pop_size)]
        self.generation = 0
        self.sort_by_fitness()

    def sort_by_fitness(self):
        self.individuals.sort(key=lambda x: x.fitness, reverse=True)

    def get_best(self):
        return self.individuals[0]

    def select_parent(self):
        # Tournament selection
        tournament_size = 5
        tournament = random.sample(self.individuals, tournament_size)
        return max(tournament, key=lambda x: x.fitness)

    def evolve(self):
        new_generation = []

        # 1. Elitism: Keep the best individuals
        new_generation.extend(self.individuals[:self.elitism_size])

        # 2. Selection, Crossover, & Mutation to fill the rest of the population
        while len(new_generation) < self.pop_size:
            parent1 = self.select_parent()
            parent2 = self.select_parent()

            child1, child2 = Individual.crossover(parent1, parent2)

            child1.mutate(self.mutation_rate)
            child2.mutate(self.mutation_rate)

            new_generation.append(child1)
            if len(new_generation) < self.pop_size:
                new_generation.append(child2)

        self.individuals = new_generation
        self.sort_by_fitness()
        self.generation += 1

### 3. Population Control
#### How it works:
* **`individuals`**: A collection of `Individual` instances representing the current generation.
* **`select_parent()`**: Performs **Tournament Selection**. It samples 5 random individuals and returns the one with the highest fitness.
* **`evolve()`**: Generates a new population. It uses **Elitism** to preserve the top performers unchanged, then repeatedly selects parents, crosses them over, and mutates offspring to fill the remaining population quota.



In [ ]:
# Define a simple maze: 0 = Path, 1 = Wall
# S = (0,0) Start, G = (5,5) Goal
simple_grid = [
    [0, 1, 0, 0, 0, 0],
    [0, 1, 0, 1, 1, 0],
    [0, 0, 0, 1, 0, 0],
    [1, 1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0]
]

maze = Maze(simple_grid, start=(0, 0), goal=(5, 5))
path_length = 25
pop_size = 100
mutation_rate = 0.08

population = Population(maze, pop_size, path_length, mutation_rate)

print(f"Initial generation best fitness: {population.get_best().fitness:.4f}")

# Run the algorithm
max_generations = 200
solved = False

for gen in range(max_generations):
    population.evolve()
    best = population.get_best()

    if best.final_position == maze.goal:
        print(f"\nGoal reached at Generation {population.generation}!")
        print(f"Best Individual Fitness: {best.fitness:.2f}")
        print(f"Path taken: {best.genes[:best.steps_taken]}")
        solved = True
        break

if not solved:
    print(f"\nFailed to reach goal within {max_generations} generations.")
    print(f"Closest ending position: {best.final_position}")

Initial generation best fitness: 1012.0000

Goal reached at Generation 1!
Best Individual Fitness: 1014.00
Path taken: ['D', 'D', 'R', 'R', 'D', 'D', 'R', 'R', 'R', 'U', 'D']


### 4. Running the Algorithm on a Simple Maze
This cell initializes a 6x6 maze, instantiates a population, and loops through the evolutionary process until either an individual finds the solution or the generation limit is hit.

In [ ]:
# Define a 10x10 maze optimized to allow steady fitness progression without extreme dead-end traps
# 0 = Path, 1 = Wall
# Start: (0, 0), Goal: (9, 9)
medium_grid = [
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
    [0, 1, 0, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 1, 0, 1, 1, 0, 1, 0],
    [1, 1, 0, 1, 0, 1, 0, 0, 1, 0],
    [0, 0, 0, 1, 0, 1, 0, 1, 1, 0],
    [0, 1, 1, 1, 0, 1, 0, 1, 0, 0],
    [0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
]

medium_maze = Maze(medium_grid, start=(0, 0), goal=(9, 9))

# Configure parameters for steady evolution
path_length_medium = 55
pop_size_medium = 500
mutation_rate_medium = 0.08

population_medium = Population(
    medium_maze,
    pop_size=pop_size_medium,
    path_length=path_length_medium,
    mutation_rate=mutation_rate_medium
)

print(f"Initial generation best fitness: {population_medium.get_best().fitness:.4f}")

# Run the evolutionary algorithm
max_generations_medium = 250
solved_medium = False

for gen in range(max_generations_medium):
    population_medium.evolve()
    best = population_medium.get_best()

    # Report progress every 10 generations
    if gen % 10 == 0 or gen == max_generations_medium - 1:
        print(f"Generation {population_medium.generation}: Best Fitness = {best.fitness:.4f}, Ending Pos = {best.final_position}")

    if best.final_position == medium_maze.goal:
        print(f"\nGoal reached at Generation {population_medium.generation}!")
        print(f"Best Individual Fitness: {best.fitness:.2f}")
        print(f"Path taken: {best.genes[:best.steps_taken]}")
        solved_medium = True
        break

if not solved_medium:
    print(f"\nFailed to reach goal within {max_generations_medium} generations.")
    print(f"Closest ending position: {best.final_position} (Distance remaining: {medium_maze.get_distance(best.final_position, medium_maze.goal)})")

Initial generation best fitness: 48.5000
Generation 1: Best Fitness = 48.5000, Ending Pos = (9, 5)

Goal reached at Generation 5!
Best Individual Fitness: 2002.00
Path taken: ['D', 'U', 'D', 'D', 'R', 'R', 'D', 'R', 'L', 'R', 'R', 'U', 'U', 'D', 'D', 'R', 'D', 'U', 'D', 'R', 'R', 'D', 'L', 'R', 'D', 'D', 'R', 'L', 'L', 'D', 'R', 'R', 'R', 'D', 'L', 'R', 'D', 'D', 'R', 'R', 'R', 'U', 'D', 'U', 'R', 'R', 'R', 'R', 'D', 'D', 'U', 'R', 'R']


### 5. Running the Algorithm on a Medium Maze
This setup evaluates the scalability of our Genetic Algorithm on a more complex 10x10 environment, requiring more steps and a larger population pool to discover a successful path.